In [1]:
import pandas as pd
import numpy as np
import json
import logging
from typing import List, Tuple, Dict, Any, Optional
import warnings
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model
from datasets import Dataset

logger = logging.getLogger(__name__)
warnings.filterwarnings('ignore')

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
file_path = "/content/drive/MyDrive/Colab Notebooks/efsa_sentiment_classification.csv"
df = pd.read_csv(file_path)

In [4]:
def format_prompt_target(row):
    """
    Given a DataFrame row, format a combined input-output string for fine-tuning.
    """
    prompt = f"[INST] Financial text: {row['Sentence']} [/INST]\n\n"

    target = (
        f"Company: {row['Company']}\n"
        f"Industry: {row['Industry']}\n"
        f"Coarse Event: {row['Coarse-Grained Event']}\n"
        f"Fine Event: {row['Fine-Grained Event']}\n"
        f"Sentiment: {row['Sentiment']}\n"
    )
    return prompt, target

In [5]:
train = df[:822]
val = df[822:939]

In [6]:
def create_dataset(df):
  inputs = []
  labels = []

  for _, row in df.iterrows():
      prompt, target = format_prompt_target(row)
      inputs.append(prompt)
      labels.append(target)

  # Create HuggingFace dataset
  hf_dataset = Dataset.from_dict({"input_text": inputs, "target_text": labels})

  return hf_dataset

train_dataset = create_dataset(train)
val_dataset = create_dataset(val)

In [7]:
from collections.abc import ValuesView
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    texts = [inp + tgt for inp, tgt in zip(examples["input_text"], examples["target_text"])]
    tokenized = tokenizer(
        texts,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    input_lens = [len(tokenizer(inp).input_ids) for inp in examples["input_text"]]
    labels = tokenized.input_ids.clone()

    for i, prompt_len in enumerate(input_lens):
        labels[i, :prompt_len] = -100

    tokenized["labels"] = labels
    return tokenized

train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)
val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=val_dataset.column_names)

Map:   0%|          | 0/822 [00:00<?, ? examples/s]

Map:   0%|          | 0/117 [00:00<?, ? examples/s]

In [8]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 129.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 113.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninst

In [8]:
from peft import prepare_model_for_kbit_training

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", load_in_8bit=True)

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [10]:
from sklearn.metrics import f1_score

def compute_metrics(eval_preds):
    predictions, labels = eval_preds

    # Convert logits to token IDs
    if predictions.ndim == 3:  # logits have shape (batch_size, seq_len, vocab_size)
        predictions = np.argmax(predictions, axis=-1)

    # Replace -100 in labels with pad_token_id for decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Clean up the decoded text
    decoded_preds = [pred.strip().lower() for pred in decoded_preds]
    decoded_labels = [label.strip().lower() for label in decoded_labels]

    def extract_field(text, field_name):
        for line in text.splitlines():
            if line.lower().startswith(field_name.lower() + ":"):
                return line.split(":", 1)[-1].strip()
        return ""

    # Extract fields from predictions and labels
    pred_fields = []
    label_fields = []

    for pred, label in zip(decoded_preds, decoded_labels):
        pred_fields.append([
            extract_field(pred, "Company"),
            extract_field(pred, "Industry"),
            extract_field(pred, "Coarse Event"),
            extract_field(pred, "Fine Event"),
            extract_field(pred, "Sentiment")
        ])
        label_fields.append([
            extract_field(label, "Company"),
            extract_field(label, "Industry"),
            extract_field(label, "Coarse Event"),
            extract_field(label, "Fine Event"),
            extract_field(label, "Sentiment")
        ])

    # Flatten the field lists for F1 calculation
    pred_flat = [field for fields in pred_fields for field in fields]
    label_flat = [field for fields in label_fields for field in fields]

    return {"macro_f1": f1_score(label_flat, pred_flat, average="macro", zero_division=0),
            "weighted_f1": f1_score(label_flat, pred_flat, average="weighted", zero_division=0)}

In [11]:
from transformers import TrainingArguments
import torch

torch.cuda.empty_cache()

training_args = TrainingArguments(
    output_dir="./lora_finetune_f1",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=10,
    learning_rate=1e-4,
    weight_decay=0.01,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=True,
    gradient_checkpointing=True,
    report_to="none",
    label_names=["labels"],
)

data_collator = DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [12]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
model.save_pretrained("/content/drive/MyDrive/Colab Notebooks/mistral_lora_model")
tokenizer.save_pretrained("/content/drive/MyDrive/Colab Notebooks/mistral_lora_tokenizer")